# nonstar — MMD vs L2 guidance, with trained models

`experiments/nonstar/nonstar.py` studies inverse design when **no design point realises the
requested target**: the joint is a 3-component GMM over `(x, y)` with components at
`(-2, 2)`, `(-2, 4)`, `(+2, 3)` (weights `1/4, 1/4, 1/2`). `x_bi = -2` is *exactly right half
the time* (bimodal conditional on `{2, 4}`); `x_uni = +2` is *never right but always close*
(`N(3, 0.2^2)`). Neither reproduces the requested point target `y* = 4`. That script's central
finding: its `mmd` arm (distributional guidance) reliably lands on `x_bi`, while its
`point_sq` arm (pointwise squared-error guidance, `(y - y*)^2`) is *provably minimised* by
`x_uni` -- worse than not guiding at all, by its own regret readout.

This notebook redoes that comparison with **actual trained models** (a conditional
Consistency Model / MLGD-F, plus the unconditional Diffusion prior it needs for guidance, in
the style of `Exp_2D_cond_1D.ipynb`) instead of nonstar.py's closed-form analytic sampler,
under one shared setup:

* **target**: a zero-sigma point mass at `y* = 4` (`sigma_t -> 0`)
* **nsamples**: 32 model + 32 target samples per guidance step (`N_MMD_BATCH`, matching
  `nonstar.py`)

and asks: does the loss function alone -- **MMD** (distributional) vs **L2**
(pointwise squared error) -- still send the optimizer to different design points, the way it
does in the closed-form script? `Optimization.optimize_LGD` gained an `L2` branch for this
(`_step_loss` in `Optimization.py`): with a degenerate (zero-sigma) target it reduces exactly
to `nonstar.py`'s `(y - y*)^2`, just batched over 32 samples/step instead of 1.


In [ ]:
import os
# ============================================================
# CONFIG
# Structure mirrors Exp_2D_cond_1D.ipynb:
#   simulations/src/        <- all .py modules
#   simulations/notebooks/  <- this notebook
#   simulations/params/     <- canonical GMM parameters (shared, load first)
#   simulations/checkpoints/
#   simulations/results/
# ============================================================
EXPERIMENT_NAME   = "nonstar"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

# QUICK_RUN trades fidelity for wall-clock time (useful to smoke-test on CPU / no GPU).
# Set False for a full-fidelity run (needs a GPU for N_ATTEMP_OPTIM=25 in reasonable time).
QUICK_RUN         = False

BASE_DIR = os.path.normpath(os.path.join(os.getcwd(), ".."))
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture -- Diffusion (unconditional prior only; no conditional Diffusion/MLGD trained)
NBLOCKS           = 3
NUNITS            = 128

# Architecture -- Consistency Model (MLGD-F, the only conditional model trained)
NBLOCKS_CM        = 3
NUNITS_CM         = 128

# Training
NEPOCHS           = 2_000  if QUICK_RUN else 20_000
BATCH_SIZE        = 1_024
NEPOCHS_CM        = NEPOCHS
BATCH_SIZE_CM     = 1_024

# Diffusion
DIFFUSION_STEPS   = 100

# GMM dimensions
CONDITION_ON      = 1   # dim(x)=1, dim(y)=1

# --- nonstar scenario (verbatim from experiments/nonstar/nonstar.py) ---
X_BI, X_UNI = -2.0, 2.0        # the two candidate design points
SIGMA_X, SIGMA_Y = 0.7, 0.2    # joint GMM component std devs (isotropic per component)
Y_STAR      = 4.0              # requested target -- a point mass (sigma_t -> 0)
ZERO_SIGMA_EPS = 1e-6          # numerical floor so the target still has an invertible covariance

# --- MMD vs L2 guidance loss ablation ---
N_MMD_BATCH = 32               # samples/step, both sides -- matches nonstar.py's N_MMD_BATCH
LOSS_ARMS   = ["MMD", "L2"]    # passed straight through to Optimization.optimize_LGD(loss=...)

# Optimization
N_ATTEMP_OPTIM     = 3   if QUICK_RUN else 25
NUM_X_T_LGD_CM     = 1   if QUICK_RUN else 3


In [ ]:
import os, sys

def _find_src_dir(start=None):
    """Walk up from the current working dir (or a Colab clone root) looking for
    a `simulations/src` directory -- robust to running this notebook from
    simulations/notebooks/ (the normal case) or from a repo root (e.g. after a
    fresh `git clone` in Colab, before doing `%cd simulations/notebooks`)."""
    candidates = [
        os.path.normpath(os.path.join(os.getcwd(), "..", "src")),                 # simulations/notebooks/ -> simulations/src
        os.path.normpath(os.path.join(os.getcwd(), "simulations", "src")),        # repo root -> simulations/src
        os.path.normpath(os.path.join(os.getcwd(), "src")),                       # already inside simulations/
        "/content/conditional-matching-paper/simulations/src",                    # common Colab clone path
    ]
    for c in candidates:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "Diffusion.py")):
            return c
    raise FileNotFoundError(
        "Could not locate simulations/src. If you're on Colab, make sure you've "
        "cloned the repo and either %cd into simulations/notebooks, or that "
        "/content/conditional-matching-paper exists."
    )

src_path = _find_src_dir()
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"src path on sys.path: {src_path}")


In [ ]:
# Install dependencies if needed
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "flow_matching", "POT", "-q"])


In [ ]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
from LossFunctions import MMDLoss, RBF

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Imports done.")


In [ ]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)


In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Joint GMM (the `nonstar` scenario)

Three components over `(x, y)`: `(-2, 2)`, `(-2, 4)`, `(+2, 3)`, weights `(1/4, 1/4, 1/2)`,
diagonal covariance `diag(sigma_x^2, sigma_y^2)` per component. `x = -2` gives a bimodal
`P(Y|X)` on `{2, 4}`; `x = +2` gives a unimodal `P(Y|X) = N(3, sigma_y^2)`. Neither matches
`N(y*=4, sigma_t^2)` exactly -- that's the "no star" (no point that hits the target) setup.

There is no `x_star` here (unlike `Exp_2D_cond_1D.ipynb`) since, by construction, no single
design point realises the requested target.


In [ ]:
GMM_PATH = os.path.join(PARAMS_DIR, f"{EXPERIMENT_NAME}_gmm_params.pt")

if os.path.exists(GMM_PATH) and not FORCE_RETRAIN:
    _saved = torch.load(GMM_PATH)
    mu_list, Sigma_list, alpha = _saved["mu_list"], _saved["Sigma_list"], _saved["alpha"]
    print(f"[GMM] Loaded from {GMM_PATH}")
else:
    mu_list = [
        torch.tensor([X_BI,  2.0]),
        torch.tensor([X_BI,  4.0]),
        torch.tensor([X_UNI, 3.0]),
    ]
    Sigma_list = [torch.diag(torch.tensor([SIGMA_X ** 2, SIGMA_Y ** 2]))] * len(mu_list)
    alpha = torch.tensor([0.25, 0.25, 0.50])

    os.makedirs(PARAMS_DIR, exist_ok=True)
    torch.save({"mu_list": mu_list, "Sigma_list": Sigma_list, "alpha": alpha}, GMM_PATH)
    print(f"[GMM] Parameters generated and saved to {GMM_PATH}")

mu_list    = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha      = alpha.float()
print(f"x_bi={X_BI}, x_uni={X_UNI}, {len(mu_list)} joint components")


## Data

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(5_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.5, s=10)
plt.axvline(X_BI, color="C1", ls="--", label=f"x_bi={X_BI}")
plt.axvline(X_UNI, color="C2", ls="--", label=f"x_uni={X_UNI}")
plt.title("Scatter Plot of P(X,Y) -- nonstar joint")
plt.xlabel("X"); plt.ylabel("Y"); plt.legend(); plt.grid(True); plt.show()


## Train Models

### Consistency Model -- P(Y|X=x)  (MLGD-F)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_checkpoint_with_hf_fallback(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )


### Diffusion -- P(X=x)  (unconditional prior, needed by the CM-guided optimizer)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )


## MMD vs L2 guidance ablation

Same zero-sigma target and `nsamples=32` for both arms; only `Optimization.optimize_LGD`'s
`loss` argument changes (`"MMD"` vs `"L2"`). The `L2` branch computes
`((target_samples - mog_samples) ** 2).mean()`: since `mog_samples` here is a degenerate
batch (every row `== y*`, `sigma_t -> 0`), this is exactly `nonstar.py`'s
`(y - y*)^2` pointwise loss, batched over 32 samples/step instead of 1.


In [ ]:
def make_target_gmm(sigma_t=0.0):
    """Zero-sigma point-mass target N(Y_STAR, sigma_t^2) -> Y_STAR, in the (mu, Sigma, w)
    tensor format dist_utils/Optimization expect (each component's covariance as a [1,1]
    matrix). sigma_t is floored at ZERO_SIGMA_EPS so the covariance stays invertible."""
    var = max(sigma_t ** 2, ZERO_SIGMA_EPS)
    mog_means     = torch.tensor([[Y_STAR]], dtype=torch.float32)
    mog_variances = torch.tensor([[[var]]], dtype=torch.float32)
    weights       = torch.tensor([1.0], dtype=torch.float32)
    return mog_means, mog_variances, weights

TARGET_MU, TARGET_S, TARGET_W = make_target_gmm()
print(f"target: N({Y_STAR}, sigma_t^2 -> {ZERO_SIGMA_EPS}), nsamples={N_MMD_BATCH}, arms={LOSS_ARMS}")


### Oracle: closed-form `L(x) = ||P(Y|X=x) - target||^2`

Same analytic-oracle convention as `Exp_2D_infeasible_targets.ipynb`: `L(x)` is the exact
closed-form GMM L2 distance (`dist_utils.gmm_l2_distance`), found by grid search then a local
refine pass. This is the same closed-form metric for both arms (MMD and L2 only change *how
the optimizer is guided*, not how landing quality is scored) -- regret is
`L(x_pred) - L*`, distance to the best achievable design point, not to zero.


In [ ]:
ORACLE_GRID_LO, ORACLE_GRID_HI, ORACLE_GRID_N, ORACLE_REFINE_N = -6.0, 6.0, 241, 81

def true_conditional_at(x_val, threshold=0.01):
    x = torch.tensor([float(x_val)])
    mu_c, S_c = dist_utils.compute_conditionals(mu_list, Sigma_list, x)
    w_c = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x)
    return dist_utils.filter_and_normalize(mu_c, S_c, w_c, threshold=threshold)

def scenario_loss(x_val, tgt_mu, tgt_S, tgt_w):
    mu_c, S_c, w_c = true_conditional_at(x_val)
    return dist_utils.gmm_l2_distance(mu_c, S_c, w_c, tgt_mu, tgt_S, tgt_w)

def oracle_grid_search(tgt_mu, tgt_S, tgt_w, lo=ORACLE_GRID_LO, hi=ORACLE_GRID_HI,
                        n=ORACLE_GRID_N, n_refine=ORACLE_REFINE_N):
    xs = torch.linspace(lo, hi, n)
    losses = torch.tensor([scenario_loss(x.item(), tgt_mu, tgt_S, tgt_w) for x in xs])
    i = int(losses.argmin())
    lo2 = xs[max(i - 2, 0)].item()
    hi2 = xs[min(i + 2, n - 1)].item()
    xs2 = torch.linspace(lo2, hi2, n_refine)
    losses2 = torch.tensor([scenario_loss(x.item(), tgt_mu, tgt_S, tgt_w) for x in xs2])
    j = int(losses2.argmin())
    return xs2[j].item(), losses2[j].item(), (xs, losses)

X_ORACLE, L_STAR, ORACLE_CURVE = oracle_grid_search(TARGET_MU, TARGET_S, TARGET_W)
print(f"x_oracle={X_ORACLE:.3f}  L*={L_STAR:.6f}")


In [ ]:
xs, losses = ORACLE_CURVE
plt.figure(figsize=(6, 4))
plt.plot(xs.numpy(), losses.numpy(), lw=1.5)
plt.axvline(X_ORACLE, color="red", ls="--", label=f"x_oracle={X_ORACLE:.2f}")
plt.axvline(X_BI, color="C1", ls=":", alpha=0.6, label="x_bi")
plt.axvline(X_UNI, color="C2", ls=":", alpha=0.6, label="x_uni")
plt.title(f"L(x) vs zero-sigma target at y*={Y_STAR}\nL*={L_STAR:.4f}")
plt.xlabel("x"); plt.ylabel("L(x)"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### Run MLGD-F (CM-guided) optimization: MMD arm vs L2 arm

Identical `Optimization.optimize_LGD` call for both arms -- only `loss` changes.


In [ ]:
def run_arm(loss_name, n_attempts):
    x_preds, l2_gmm_list, regret_list, times, final_losses = [], [], [], [], []
    for i in trange(n_attempts, desc=loss_name, leave=False):
        experiment_utils.set_run_seed(GLOBAL_SEED, i)
        t0 = time.time()
        best_x_t, _, final_loss = Optimization.optimize_LGD(
            model_uncond, Cos_ConsistencyModeliCT,
            TARGET_MU, TARGET_S, TARGET_W,
            mu_list, Sigma_list, alpha,
            nsamples=N_MMD_BATCH, loss=loss_name, device=device,
            CM=True, FLAG=False, num_x_t=NUM_X_T_LGD_CM,
        )
        times.append(time.time() - t0)
        final_losses.append(final_loss.item() if hasattr(final_loss, "item") else final_loss)
        x_preds.append(best_x_t)

        x_pred = best_x_t.float().view(-1).cpu()
        mu_p, S_p, w_p = true_conditional_at(x_pred.item())
        l2_gmm = dist_utils.gmm_l2_distance(mu_p, S_p, w_p, TARGET_MU, TARGET_S, TARGET_W)

        l2_gmm_list.append(l2_gmm)
        regret_list.append(l2_gmm - L_STAR)
        print(f"[{loss_name}][{i+1}/{n_attempts}] x_pred={x_pred.item():+.3f}  "
              f"L2_GMM={l2_gmm:.4f}  regret={l2_gmm - L_STAR:.4f}")
    return dict(x_pred=[x.detach().cpu() for x in x_preds], l2_gmm=l2_gmm_list,
                regret=regret_list, times=times, final_loss=final_losses)

all_results = {}
for loss_name in LOSS_ARMS:
    print(f"\n=== arm={loss_name} (nsamples={N_MMD_BATCH}, target=N({Y_STAR}, sigma_t->0), L*={L_STAR:.6f}) ===")
    all_results[loss_name] = run_arm(loss_name, N_ATTEMP_OPTIM)


## Results summary

In [ ]:
def landing_side(x):
    return "x_bi" if abs(x - X_BI) < abs(x - X_UNI) else "x_uni"

rows = []
for loss_name in LOSS_ARMS:
    res = all_results[loss_name]
    l2_gmm = np.array(res["l2_gmm"]); regret = np.array(res["regret"]); times = np.array(res["times"])
    x_preds = np.array([x.item() for x in res["x_pred"]])
    pct_bi = 100.0 * np.mean([landing_side(x) == "x_bi" for x in x_preds])
    rows.append({
        "Arm":          loss_name,
        "L* (oracle)":  f"{L_STAR:.4f}",
        "x_oracle":     f"{X_ORACLE:.3f}",
        "Regret":       f"{regret.mean():.4f} +/- {regret.std():.4f}",
        "% at x_bi":    f"{pct_bi:.0f}%",
        "Time (s)":     f"{times.mean():.2f} +/- {times.std():.2f}",
        "N restarts":   len(l2_gmm),
    })
df = pd.DataFrame(rows).set_index("Arm")
display(df)


## Save results

In [ ]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

out = {
    "experiment": EXPERIMENT_NAME,
    "seed": GLOBAL_SEED,
    "environment": env_info,
    "quick_run": QUICK_RUN,
    "scenario": {
        "x_bi": X_BI, "x_uni": X_UNI, "sigma_x": SIGMA_X, "sigma_y": SIGMA_Y,
        "y_star": Y_STAR, "zero_sigma_eps": ZERO_SIGMA_EPS,
    },
    "meta": {
        "nsamples": N_MMD_BATCH,
        "n_attemp_optim": N_ATTEMP_OPTIM,
        "num_x_t_lgd_cm": NUM_X_T_LGD_CM,
        "nepochs": NEPOCHS,
        "x_oracle": X_ORACLE,
        "L_star": L_STAR,
    },
    "arms": {},
}
for loss_name in LOSS_ARMS:
    res = all_results[loss_name]
    out["arms"][loss_name] = {
        "x_pred": [to_python(x) for x in res["x_pred"]],
        "final_loss": [to_python(v) for v in res["final_loss"]],
        "l2_gmm": [to_python(v) for v in res["l2_gmm"]],
        "regret": [to_python(v) for v in res["regret"]],
        "times": res["times"],
    }

suffix = "_quickrun" if QUICK_RUN else ""
path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_mmd_vs_l2_results_seed{GLOBAL_SEED}{suffix}.json")
with open(path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Results saved to {path}")
